# 01 - Tokenization and Embeddings

## Imports

In [1]:
from collections import defaultdict
import numpy as np

# Make sure we get outputs from a probabilitistic model each time (for reproducibility)
np.random.seed(1515)

## Why Not Just Split on Spaces?

`ai_resources/ai_primer/01-context-is-key.md` already told you tokens are the billing unit and pointed you at the OpenAI Tokenizer tool to watch text get split into them. This notebook builds the thing that does the splitting.

The simplest option - one token per word - has a real problem: `climb`, `climbing`, and `climbed` would become three totally unrelated vocabulary entries, sharing nothing, even though they obviously share meaning. **Byte-Pair Encoding (BPE)** fixes this by learning *sub-word* pieces from data - common roots like `climb` end up as their own token, shared across every word that contains them, purely because they show up together often enough to be worth merging.

## Byte-Pair Encoding, From Scratch

BPE starts with every word broken into individual characters (plus an end-of-word marker, so `climb` doesn't get confused with the identical characters inside `climbed` before the first merge happens), then repeatedly merges whichever *adjacent pair* of symbols is currently most frequent across the whole corpus - one merge per round.

In [2]:
corpus = [
    "climb", "climbing", "climbed", "climb", "climbing",
    "auto", "autonomous", "auto",
    "score", "scoring", "scored", "score", "scoring",
    "robot", "robots", "robot",
    "team", "teams", "team",
    "scout", "scouting", "scouted", "scout",
]

word_freq = defaultdict(int)
for word in corpus:
    word_freq[word] += 1

END = "</w>"   # marks the end of a word, so 'ing' at a word's end isn't confused with
                # 'ing' appearing mid-word
vocab = {tuple(list(word) + [END]): freq for word, freq in word_freq.items()}
print("Starting vocabulary (one entry per word, split into characters):")
for symbols, freq in vocab.items():
    print(f"  {symbols}  (freq={freq})")

Starting vocabulary (one entry per word, split into characters):
  ('c', 'l', 'i', 'm', 'b', '</w>')  (freq=2)
  ('c', 'l', 'i', 'm', 'b', 'i', 'n', 'g', '</w>')  (freq=2)
  ('c', 'l', 'i', 'm', 'b', 'e', 'd', '</w>')  (freq=1)
  ('a', 'u', 't', 'o', '</w>')  (freq=2)
  ('a', 'u', 't', 'o', 'n', 'o', 'm', 'o', 'u', 's', '</w>')  (freq=1)
  ('s', 'c', 'o', 'r', 'e', '</w>')  (freq=2)
  ('s', 'c', 'o', 'r', 'i', 'n', 'g', '</w>')  (freq=2)
  ('s', 'c', 'o', 'r', 'e', 'd', '</w>')  (freq=1)
  ('r', 'o', 'b', 'o', 't', '</w>')  (freq=2)
  ('r', 'o', 'b', 'o', 't', 's', '</w>')  (freq=1)
  ('t', 'e', 'a', 'm', '</w>')  (freq=2)
  ('t', 'e', 'a', 'm', 's', '</w>')  (freq=1)
  ('s', 'c', 'o', 'u', 't', '</w>')  (freq=2)
  ('s', 'c', 'o', 'u', 't', 'i', 'n', 'g', '</w>')  (freq=1)
  ('s', 'c', 'o', 'u', 't', 'e', 'd', '</w>')  (freq=1)


In [3]:
def get_pair_counts(vocab):
    """Count every adjacent symbol pair across the whole corpus, weighted by word frequency."""
    pair_counts = defaultdict(int)
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pair_counts[(symbols[i], symbols[i + 1])] += freq
    return pair_counts

def merge_vocab(pair, vocab):
    """Replace every occurrence of `pair` with a single merged symbol."""
    merged_symbol = pair[0] + pair[1]
    new_vocab = {}
    for symbols, freq in vocab.items():
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        new_vocab[tuple(new_symbols)] = freq
    return new_vocab

In [4]:
n_merges = 15
merges = []

for step in range(n_merges):
    pair_counts = get_pair_counts(vocab)
    if not pair_counts:
        break
    best_pair = max(pair_counts, key=pair_counts.get)
    vocab = merge_vocab(best_pair, vocab)
    merges.append(best_pair)
    print(f"merge {step + 1:2d}: {best_pair} (seen {pair_counts[best_pair]} times) "
          f"-> '{best_pair[0] + best_pair[1]}'")

merge  1: ('s', 'c') (seen 9 times) -> 'sc'
merge  2: ('sc', 'o') (seen 9 times) -> 'sco'
merge  3: ('u', 't') (seen 7 times) -> 'ut'
merge  4: ('c', 'l') (seen 5 times) -> 'cl'
merge  5: ('cl', 'i') (seen 5 times) -> 'cli'
merge  6: ('cli', 'm') (seen 5 times) -> 'clim'
merge  7: ('clim', 'b') (seen 5 times) -> 'climb'
merge  8: ('i', 'n') (seen 5 times) -> 'in'
merge  9: ('in', 'g') (seen 5 times) -> 'ing'
merge 10: ('ing', '</w>') (seen 5 times) -> 'ing</w>'
merge 11: ('sco', 'r') (seen 5 times) -> 'scor'
merge 12: ('sco', 'ut') (seen 4 times) -> 'scout'
merge 13: ('e', 'd') (seen 3 times) -> 'ed'
merge 14: ('ed', '</w>') (seen 3 times) -> 'ed</w>'
merge 15: ('a', 'ut') (seen 3 times) -> 'aut'


Watch which merges happen early: `climb` assembles itself one character pair at a time purely because it's the most frequent substring around, long before the algorithm ever "knows" it's a meaningful English root. Same for `scor` and `scout`, and for the suffixes `ing</w>` and `ed</w>` - shared endings earn their own tokens too, since they show up across multiple different words.

In [5]:
print("Final vocabulary (each word as a sequence of learned sub-word tokens):")
for symbols, freq in vocab.items():
    print(f"  {symbols}  (freq={freq})")

Final vocabulary (each word as a sequence of learned sub-word tokens):
  ('climb', '</w>')  (freq=2)
  ('climb', 'ing</w>')  (freq=2)
  ('climb', 'ed</w>')  (freq=1)
  ('aut', 'o', '</w>')  (freq=2)
  ('aut', 'o', 'n', 'o', 'm', 'o', 'u', 's', '</w>')  (freq=1)
  ('scor', 'e', '</w>')  (freq=2)
  ('scor', 'ing</w>')  (freq=2)
  ('scor', 'ed</w>')  (freq=1)
  ('r', 'o', 'b', 'o', 't', '</w>')  (freq=2)
  ('r', 'o', 'b', 'o', 't', 's', '</w>')  (freq=1)
  ('t', 'e', 'a', 'm', '</w>')  (freq=2)
  ('t', 'e', 'a', 'm', 's', '</w>')  (freq=1)
  ('scout', '</w>')  (freq=2)
  ('scout', 'ing</w>')  (freq=1)
  ('scout', 'ed</w>')  (freq=1)


## Encoding a Word the Tokenizer Has Never Seen

The real test of a learned tokenizer isn't the training words - it's what happens to a word that was never in the corpus at all. Applying the merges *in the order they were learned* to a new word's characters is exactly how a real BPE tokenizer encodes new text.

In [6]:
def encode_word(word, merges):
    symbols = list(word) + [END]
    for pair in merges:
        merged_symbol = pair[0] + pair[1]
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols

for test_word in ["climbable", "scoreboard", "roboticist", "scoutmaster", "unseenword"]:
    print(f"{test_word:15s} -> {encode_word(test_word, merges)}")

climbable       -> ['climb', 'a', 'b', 'l', 'e', '</w>']
scoreboard      -> ['scor', 'e', 'b', 'o', 'a', 'r', 'd', '</w>']
roboticist      -> ['r', 'o', 'b', 'o', 't', 'i', 'c', 'i', 's', 't', '</w>']
scoutmaster     -> ['scout', 'm', 'a', 's', 't', 'e', 'r', '</w>']
unseenword      -> ['u', 'n', 's', 'e', 'e', 'n', 'w', 'o', 'r', 'd', '</w>']


`climbable` and `scoreboard` correctly reuse the learned `climb` and `scor` tokens, even though neither word was ever in the training corpus. `roboticist` and `unseenword` share little or nothing with the training data, so they fall back to mostly individual characters - exactly what should happen when a tokenizer meets something genuinely unfamiliar, rather than silently failing. Real tokenizers (GPT's `tiktoken`, SentencePiece) run this exact algorithm on a training corpus of billions of words and tens of thousands of merges instead of 15 - the mechanism doesn't change, only the scale.

## From Tokens to Vectors: Embeddings

A token is still just an ID at this point - an index into the vocabulary, not something a neural network can do arithmetic on directly. An **Embedding Matrix** fixes this: one row per vocabulary token, each row a learned vector. Looking up a token's embedding is exactly the row-indexing you'd expect from `linear_algebra_primer/02-matrices-and-transformations.ipynb` - nothing more exotic than picking out one row of a matrix.

In [7]:
vocabulary = sorted({symbol for symbols in vocab.keys() for symbol in symbols})
token_to_id = {token: i for i, token in enumerate(vocabulary)}
print(f"Vocabulary size: {len(vocabulary)}")
print(f"Vocabulary: {vocabulary}")

embedding_dim = 4   # tiny, just to keep the printed vectors readable
embedding_matrix = np.random.randn(len(vocabulary), embedding_dim) * 0.5
print(f"\nEmbedding matrix shape: {embedding_matrix.shape}  (vocab_size x embedding_dim)")

Vocabulary size: 17
Vocabulary: ['</w>', 'a', 'aut', 'b', 'climb', 'e', 'ed</w>', 'ing</w>', 'm', 'n', 'o', 'r', 's', 'scor', 'scout', 't', 'u']

Embedding matrix shape: (17, 4)  (vocab_size x embedding_dim)


In [8]:
def embed_tokens(tokens, token_to_id, embedding_matrix):
    ids = [token_to_id[t] for t in tokens]
    return embedding_matrix[ids]   # row-indexing - the entire "embedding lookup"

tokens = encode_word("climbing", merges)
vectors = embed_tokens(tokens, token_to_id, embedding_matrix)
print(f"Tokens: {tokens}")
print(f"Embedding vectors (one {embedding_dim}-dim row per token):")
print(vectors.round(3))

Tokens: ['climb', 'ing</w>']
Embedding vectors (one 4-dim row per token):
[[-0.663 -0.231  0.238 -0.599]
 [ 0.233  0.091  1.062  0.591]]


That's the entire embedding step: a sequence of tokens becomes a sequence of vectors, just by indexing rows out of one matrix. These random vectors don't mean anything yet - in a real model, this matrix is *learned* during training, so that tokens used in similar contexts end up with similar vectors. What happens to this sequence of vectors next - how the model actually relates one token's vector to another's - is exactly where `02-attention-mechanism.ipynb` picks up.

## Try It Yourself

Add a few more scouting-flavored words to `corpus` that share roots with the existing ones but weren't there before - e.g. `climbs`, `scores`, `robotic` - and re-run the BPE training loop from scratch. Then encode a brand-new word that combines two roots you've now trained on, like `roboticscore` (not a real word, that's fine) - does it decompose into the two learned root tokens, or fall back to characters? What does that tell you about how much shared structure BPE actually needs to see before it commits a pattern to the vocabulary?

In [9]:
# TODO: add climbs, scores, robotic (or similar) to the corpus, rerun BPE training,
# and encode a new made-up word combining two learned roots


## Resources

- Sennrich, R., Haddow, B., & Birch, A. (2016). "Neural Machine Translation of Rare Words with Subword Units." *ACL 2016*. The original paper introducing BPE for language models - the exact algorithm implemented above (paper).
- [OpenAI Tokenizer](https://platform.openai.com/tokenizer) - already linked from `ai_primer/01-context-is-key.md`; paste in text and compare a real, production BPE tokenizer's output to what this notebook's toy version would do (tool).
- [Hugging Face: Byte-Pair Encoding Tokenization](https://huggingface.co/learn/nlp-course/chapter6/5) - a clear walkthrough of the same algorithm, with the real edge cases (unicode, byte-level fallbacks) this notebook simplified away (instructional).